**Evaluating a Boolean circuit with matrix algebra.**
This notebook applies the gate-matrix formalism from `gate_matrices.ipynb`
to a concrete four-input circuit.
The circuit implements:

$$\text{out} = \lnot\,\bigl[(\,a \land b\,) \lor (\lnot c)\bigr] \land d$$

Each gate is a matrix; each Boolean value is a one-hot column vector.
The Kronecker product $\mathbf{x} \otimes \mathbf{y}$ packs two one-hot
vectors into the four-element input that AND and OR expect,
and **composing gates is matrix multiplication**.
Because a one-hot vector stores its value in the second row,
reading `v[1, 0]` recovers the bit 0 or 1 directly.
The circuit is evaluated on a single input, then across the
full 16-row truth table.

In [ ]:
"""simple_circuit.ipynb"""

# Cell 01 - Imports and common Boolean states and gates

import numpy as np
from IPython.display import display

from qis101_utils import as_latex

# One-hot (indicator) vectors: the position of the 1 encodes the state
# F = |0⟩ = [1, 0]ᵀ  (index 0 = False)
# T = |1⟩ = [0, 1]ᵀ  (index 1 = True)
f = np.array([[1], [0]])
t = np.array([[0], [1]])

g_not = np.array([[0, 1], [1, 0]])
g_and = np.array([[1, 1, 1, 0], [0, 0, 0, 1]])
g_or = np.array([[1, 0, 0, 0], [0, 1, 1, 1]])

display(as_latex(f, prefix=r"\mathbf{F}=0="))
display(as_latex(t, prefix=r"\mathbf{T}=1="))
display(as_latex(g_not, prefix=r"\mathbf{NOT}="))
display(as_latex(g_and, prefix=r"\mathbf{AND}="))
display(as_latex(g_or, prefix=r"\mathbf{OR}="))

---
**Reading the bit back out of a one-hot vector.**

Every Boolean value in this notebook is a $2 \times 1$ column vector,
so `circuit()` returns a vector rather than a plain `0` or `1`.
To print a result we index into it with `v[1, 0]`, which means
row `1` and column `0`.

Both numbers are zero-based, so:

- Row `1` is the **second** row of the vector.
- Column `0` is the **only** column, since the vector is one element wide.

That single location is all we need to look at, because the two states
differ precisely there:

| State | Vector | Row 0 | Row 1 | `v[1, 0]` |
|---|---|---|---|---|
| False | $\begin{bmatrix} 1 \\ 0 \end{bmatrix}$ | 1 | 0 | **0** |
| True | $\begin{bmatrix} 0 \\ 1 \end{bmatrix}$ | 0 | 1 | **1** |

The 1 sits in row 0 for false and in row 1 for true.
So row 1 holds a 0 exactly when the value is false, and a 1 exactly
when the value is true, which is the bit we want to display.
Reading `v[1, 0]` is the whole conversion and no helper function is needed.

Note that `v[1]` alone is not the same thing.
It selects the row and hands back an array of shape `(1,)`,
which prints as `[0]` with brackets.
Supplying both indices reaches the single element inside and prints a bare `0`.

In [ ]:
# Cell 02 - Matrix evaluation of the circuit


def circuit(a, b, c, d):
    g1 = g_and @ np.kron(a, b)
    g2 = g_not @ c
    g3 = g_or @ np.kron(g1, g2)
    g4 = g_and @ np.kron(g3, d)
    g5 = g_not @ g4
    return g5


# Quick check that circuit works as expected
out = circuit(t, t, t, t)
print(f"circuit(1, 1, 1, 1) = {out[1, 0]}  (expected 0)")


---
**Building the truth table.**

A single call to `circuit()` tells us the output for one set of inputs.
To describe the circuit completely we need the output for *every*
set of inputs.

Each of the four inputs can be false or true, so there are
$2^4 = 16$ combinations to check.
The cell below reaches all 16 with four nested loops, one per input.
Each loop runs over the list `[f, t]`, the two one-hot vectors from Cell 01,
so the innermost body executes once for every combination.

For each combination the code:

1. Calls `circuit(a, b, c, d)` to get the output vector.
2. Prints the four input bits and the output bit, reading each one
   with the `v[1, 0]` index explained above.

The loop order fixes the row order.
Because `d` is the innermost loop it changes fastest, and `a` is the
outermost so it changes slowest.
The result is that the four input columns count upward in binary
from `0000` to `1111`, which is why the table looks sorted.

Reading the finished table, the `out` column is 0 on exactly the rows
where $(\,a \land b\,) \lor (\lnot c)$ is true and `d` is also true,
since the final NOT gate inverts that result.

In [ ]:
# Cell 03 - Full truth table across all 16 input combinations

print(f"{'a':>3} {'b':>3} {'c':>3} {'d':>3}  ->  out")
print("-" * 25)
for a in [f, t]:
    for b in [f, t]:
        for c in [f, t]:
            for d in [f, t]:
                out = circuit(a, b, c, d)
                print(
                    f"  {a[1, 0]}   {b[1, 0]}   {c[1, 0]}   {d[1, 0]}  ->   {out[1, 0]}"
                )